# University of Engineering and Technology Peshawar, Nowshera Campus
Name: MUNSIFF ALI

Registration Number: 22JZELE0489
# Lab 11: 1D CNN for Time-Series Forecasting

 
**Topic:** One-dimensional convolutional neural network for AEP forecasting

## Lab Objective
The objective of this lab is to train a 1D CNN on time-series windows. The model uses convolution layers to learn local temporal patterns and predict the target value.


## 1. Set Working Directory
The working directory is changed so the notebook can access the dataset, helper functions, checkpoints, and saved history files.


In [2]:
import os
os.chdir(r'C:\Users\Admin\Downloads\labs\New folder\Code')

## 2. Import Libraries and Utilities
This cell imports forecasting metrics, sequence preparation functions, Keras layers, callbacks, plotting tools, and scaling utilities.


In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

## 3. Define Initial Parameters
The notebook initializes the model state, start epoch, number of time steps, and number of features for the 1D CNN input.


In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

## 4. Define the 1D CNN Architecture
The CNN uses 1D convolution layers to extract temporal features from a sequence window, then flattens those features and predicts one output value.


In [5]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

## 5. Display Model Summary and Diagram
The summary and model plot confirm the CNN layer sequence, tensor shapes, and number of trainable parameters.


In [6]:
model1 = CNN()
model1.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 conv1d (Conv1D)             (None, 23, 16)            688       
                                                                 
 conv1d_1 (Conv1D)           (None, 22, 16)            528       
                                                                 
 flatten (Flatten)           (None, 352)               0         
                                                                 
 dense (Dense)               (None, 1)                 353       
                                                                 
Total params: 1,569
Trainable params: 1,569
Non-trainable params: 0
_________________________________________________________________


In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


## 6. Set Checkpoint and History Paths
These paths control where the best model checkpoint, history plot, and history JSON file are saved.


In [8]:
checkpoints = r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Admin\Downloads\labs\New folder\Code'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

## 7. Configure Callbacks
Callbacks save the best model according to validation loss and record the training history.


In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## 8. Compile or Load the CNN
A new CNN is compiled when no checkpoint is supplied. If a checkpoint is provided, training can continue from the saved model.


In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 9. Load Dataset Files
The train, validation, and test CSV files are loaded and converted into arrays for sequence generation. The scaler is loaded for inverse transformation during evaluation.


In [11]:
import os
path_dataset =r'C:\Users\Admin\Downloads\labs\New folder\Code'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Admin\anaconda3\envs\ML\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## 10. Prepare Time-Series Input Windows
The time step and feature count are confirmed, then the data is converted into supervised-learning sequences for training, validation, and testing.


In [12]:
time_steps=24
num_features=21

In [13]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.0070002079010009766 sec


## 11. Train the 1D CNN
The CNN is trained using the prepared sequence windows and monitored with validation data after each epoch.


In [14]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10
24/27 [=========================>....] - ETA: 0s - loss: 0.2517 - mae: 0.2517 - mape: 185.4080
Epoch 1: val_loss improved from inf to 0.11473, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5
27/27 [==============================] - 5s 61ms/step - loss: 0.2391 - mae: 0.2391 - mape: 173.1906 - val_loss: 0.1147 - val_mae: 0.1147 - val_mape: 39.9443
Epoch 2/10
27/27 [==============================] - ETA: 0s - loss: 0.0949 - mae: 0.0949 - mape: 64.6903
Epoch 2: val_loss improved from 0.11473 to 0.11099, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0002-loss0.11.h5
27/27 [==============================] - 1s 48ms/step - loss: 0.0949 - mae: 0.0949 - mape: 64.6903 - val_loss: 0.1110 - val_mae: 0.1110 - val_mape: 43.5372
Epoch 3/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0746 - mae: 0.0746 - mape: 49.6833
Epoch 3: val_loss improved from 0.11099 to 0.09621, saving model to C:\Users\Admin\Downloads\labs\New

## 12. Evaluate Test Performance
The saved CNN checkpoint predicts test values. Predictions are inverse-transformed and evaluated using MAE, RMSE, MAPE, and R2.


In [16]:

model = load_model(r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 309ms/step
Mean Absolute Error (MAE): 4389.45
Median Absolute Error (MedAE): 4846.8
Mean Squared Error (MSE): 20076338.9
Root Mean Squared Error (RMSE): 4480.66
Mean Absolute Percentage Error (MAPE): 28.13 %
Median Absolute Percentage Error (MDAPE): 31.32 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## 13. Fine-Tuning Setup
This section selects a previous checkpoint and starting epoch so the model can continue training.


In [17]:
checkpoints = r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5'
model=r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5'
start_epoch= 58

## 14. Rebuild Callbacks and Load Checkpoint
The model and callbacks are prepared again for the fine-tuning stage.


In [18]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


## 15. Continue Training
Additional epochs are used to improve or refine the CNN model performance.


In [19]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
25/27 [==========================>...] - ETA: 0s - loss: 0.0929 - mae: 0.0929 - mape: 60.4470
Epoch 1: val_loss improved from inf to 0.10304, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5
27/27 [==============================] - 4s 98ms/step - loss: 0.0927 - mae: 0.0927 - mape: 61.1382 - val_loss: 0.1030 - val_mae: 0.1030 - val_mape: 37.3154
Epoch 2/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0878 - mae: 0.0878 - mape: 62.8594
Epoch 2: val_loss did not improve from 0.10304
27/27 [==============================] - 2s 59ms/step - loss: 0.0874 - mae: 0.0874 - mape: 62.1552 - val_loss: 0.1105 - val_mae: 0.1105 - val_mape: 41.9352
Epoch 3/10
23/27 [========================>.....] - ETA: 0s - loss: 0.0843 - mae: 0.0843 - mape: 67.1085
Epoch 3: val_loss improved from 0.10304 to 0.09799, saving model to C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5
27/27 [==============================] - 1s 31ms/step - 

## 16. Evaluate Fine-Tuned CNN
The fine-tuned checkpoint is evaluated on the same test set so its performance can be compared with the earlier model.


In [20]:

model = load_model(r'C:\Users\Admin\Downloads\labs\New folder\Code\E1-cp-0001-loss0.11.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 205ms/step
Mean Absolute Error (MAE): 3381.25
Median Absolute Error (MedAE): 3649.97
Mean Squared Error (MSE): 12593055.66
Root Mean Squared Error (RMSE): 3548.67
Mean Absolute Percentage Error (MAPE): 21.71 %
Median Absolute Percentage Error (MDAPE): 23.58 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Conclusion
This lab shows that 1D CNNs can learn temporal patterns from fixed-length time-series windows. Checkpointing and fine-tuning help preserve the best model and continue improving performance.
